# RSNA 2026 Knee Abnormality Detection | Kaggle Submission Notebook

**A complete, offline-capable image-based baseline** for predicting 12 knee MRI findings. This notebook reads DICOM series, selects sagittal/coronal/axial images, trains a small 2.5D CNN, evaluates available labels, predicts the test set and writes `/kaggle/working/submission.csv`.

**Important:** Only a small subset of images has expert-reviewed labels. Optional English report-derived weak labels can supplement training, but they are noisy. The test reports are unavailable and are **never** used as model inputs. The notebook does not require internet access or pretrained weights.

**Kaggle settings:** Attach the RSNA Knee Abnormality Detection competition dataset, choose GPU if available, and turn **Internet OFF**. To submit, use **Save Version → Save & Run All**, then **Submit to Competition**. This is a starter model, not a claimed competitive score.

In [ ]:
import os, glob, random, re, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
warnings.filterwarnings('ignore', category=UserWarning)
try:
    import pydicom
except ImportError as e:
    raise ImportError('pydicom is required. Select a Kaggle Python image that includes pydicom, or attach an offline pydicom wheel.') from e

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
LABELS=['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
print('PyTorch',torch.__version__,'| device:',DEVICE,'| pydicom:',pydicom.__version__)

## 1. Configuration

`FAST_DEV_RUN=True` first checks the full training and submission pipeline using a limited subset. Set it to `False` for a longer run. Validation AUC may be undefined when a target has only one class in the holdout split.

In [ ]:
FAST_DEV_RUN = False   # Change to True for a quick smoke test
USE_WEAK_LABELS = True  # English report keyword weak supervision; disable to use expert labels only
IMAGE_SIZE = 128
EPOCHS = 3 if FAST_DEV_RUN else 8
BATCH_SIZE = 8 if DEVICE == 'cuda' else 4
MAX_TRAIN_STUDIES = 100 if FAST_DEV_RUN else 1800
MAX_TEST_STUDIES = None   # ALWAYS keep None for submissions. Never limit test rows.
LR = 3e-4
NUM_WORKERS = 2
MAX_FILES_PER_SERIES = 3  # uniformly sampled slices from the chosen series
TRAIN_SECONDS_BUDGET = 5.5 * 3600  # protects 9-hour competition notebook limit
OUT_DIR=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()

candidates=[Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'),
            Path('/kaggle/input/rsna-knee-abnormality-detection')]
ROOT=next((p for p in candidates if (p/'train.csv').is_file()),None)
if ROOT is None:
    for p in Path('/kaggle/input').rglob('train_series.csv') if Path('/kaggle/input').exists() else []:
        if (p.parent/'train.csv').exists() and (p.parent/'sample_submission.csv').exists():
            ROOT=p.parent; break
if ROOT is None:
    raise FileNotFoundError('Attach the RSNA Knee Abnormality Detection competition data to this notebook first.')
print('Dataset:',ROOT)

## 2. Load competition metadata and inspect targets

In [ ]:
train=pd.read_csv(ROOT/'train.csv',dtype={'StudyInstanceUID':str})
train_series=pd.read_csv(ROOT/'train_series.csv',dtype={'StudyInstanceUID':str,'SeriesInstanceUID':str})
test=pd.read_csv(ROOT/'test.csv',dtype={'StudyInstanceUID':str})
test_series=pd.read_csv(ROOT/'test_series.csv',dtype={'StudyInstanceUID':str,'SeriesInstanceUID':str})
sample=pd.read_csv(ROOT/'sample_submission.csv',dtype={'StudyInstanceUID':str})
assert all(c in train.columns for c in LABELS), 'Training label columns missing'
assert list(sample.columns)==['StudyInstanceUID']+LABELS,'Unexpected submission schema'
print('Train studies:',len(train),'| Train series:',len(train_series),'| Visible test studies:',len(test))
print('Expert labels per condition:')
display(train[LABELS].notna().sum().to_frame('Count').T)
assert train.StudyInstanceUID.is_unique and test.StudyInstanceUID.is_unique

## 3. Optional report-derived supervision

Conservative English keyword heuristics produce positive or negative labels only for explicit phrases. These are imperfect and can be wrong, especially for multilingual reports and negated findings. Missing weak labels remain missing and never count toward the loss. Expert-reviewed labels always take precedence. Review extracted labels before using them in serious experiments.

In [ ]:
PATTERNS = {
 'ACL':r'\b(?:acl|anterior cruciate ligament)\b',
 'MCL':r'\b(?:mcl|medial collateral ligament)\b',
 'Medial Meniscus':r'\bmedial menisc(?:us|al)\b',
 'Lateral Meniscus':r'\blateral menisc(?:us|al)\b',
 'Medial OA':r'\b(?:medial compartment|medial tibiofemoral)\b',
 'Lateral OA':r'\b(?:lateral compartment|lateral tibiofemoral)\b',
 'PF OA':r'\b(?:patellofemoral|patello-femoral)\b',
 'Effusion':r'\b(?:effusion|joint fluid)\b',
 'Synovitis':r'\bsynovit(?:is|ic)\b',
 "Baker's":r"\b(?:baker'?s?|popliteal) cyst\b",
 'Contusion':r'\b(?:bone contusion|bone bruise|marrow edema|marrow oedema)\b',
 'Fracture':r'\bfractur(?:e|ed)\b',
}
POSITIVE_MODIFIERS=r'\b(?:tear|tearing|torn|rupture|ruptured|injury|injured|sprain|degeneration|degenerative|osteoarthritis|arthrosis|chondrosis|chondromalacia|narrowing|present|moderate|large|small|trace|mild|severe|partial|complete|complex|horizontal|radial|displaced|edema|oedema)\b'
NEGATIVE_MODIFIERS=r'\b(?:no|without|absent|negative for|intact|normal|unremarkable|no evidence of|not seen|no signs of)\b'

def weak_label(report, label):
    if not isinstance(report,str) or len(report.strip())<8: return np.nan
    # Sentence-level only, avoiding an unrelated negation elsewhere in the report.
    sentences=re.split(r'[.\n;]+',report.lower())
    found=[]
    for sent in sentences:
        for m in re.finditer(PATTERNS[label],sent,flags=re.I):
            context=sent[max(0,m.start()-65):min(len(sent),m.end()+65)]
            prefix=sent[max(0,m.start()-42):m.start()]
            # 'no evidence of ...', 'intact ACL', etc. Rule is intentionally conservative.
            neg=bool(re.search(NEGATIVE_MODIFIERS+r'[^,]{0,35}$',prefix)) or bool(re.search(r'\b(?:intact|normal|unremarkable|without tear|without injury)\b',context))
            pos=bool(re.search(POSITIVE_MODIFIERS,context))
            if label in ['Effusion','Synovitis',"Baker's",'Contusion','Fracture']:
                pos=True  # named finding itself is a positive cue unless explicitly negated
            if neg: found.append(0.0)
            elif pos: found.append(1.0)
    if not found or (0.0 in found and 1.0 in found): return np.nan
    return float(found[0])

train_targets=train[['StudyInstanceUID']+LABELS].copy()
expert_matrix=train[LABELS].apply(pd.to_numeric,errors='coerce').to_numpy(dtype=np.float32)
values=expert_matrix.copy()
weak_mask=np.zeros_like(values,dtype=bool)
if USE_WEAK_LABELS and 'Report' in train.columns:
    for j,label in enumerate(tqdm(LABELS,desc='Extracting weak labels')):
        weak=np.array([weak_label(r,label) for r in train['Report']],dtype=np.float32)
        fill=np.isnan(values[:,j]) & np.isfinite(weak)
        values[fill,j]=weak[fill]
        weak_mask[:,j]=fill
print('Expert labels:',int(np.isfinite(expert_matrix).sum()),
      '| Additional weak labels:',int(weak_mask.sum()),
      '| Studies with at least one label:',int(np.isfinite(values).any(axis=1).sum()))
train_targets[LABELS]=values

## 4. Split studies and build DICOM series index

The validation split is study-level, so slices from one study cannot appear in both sets.

In [ ]:
all_ids=np.where(np.isfinite(values).any(axis=1))[0]
if len(all_ids)<3: raise ValueError('Not enough labeled training studies.')
train_idx,val_idx=train_test_split(all_ids,test_size=0.15,random_state=SEED)
# Prioritize expert-reviewed rows so FAST_DEV_RUN still includes reliable examples.
train_idx=sorted(train_idx,key=lambda i: (-int(np.isfinite(expert_matrix[i]).sum()),str(train.iloc[i].StudyInstanceUID)))
if MAX_TRAIN_STUDIES is not None: train_idx=train_idx[:MAX_TRAIN_STUDIES]
tr_df=train_targets.iloc[train_idx].reset_index(drop=True)
va_df=train_targets.iloc[val_idx].reset_index(drop=True)
print('Train:',len(tr_df),'| Validation:',len(va_df))

def index_series(frame):
    d={}
    for uid,group in frame.groupby('StudyInstanceUID',sort=False):
        chosen={}
        for plane in ('Sagittal','Coronal','Axial'):
            subset=group.loc[group['Anatomical_Plane'].astype(str).str.lower()==plane.lower()]
            if len(subset):
                subset=subset.sort_values(['Fluid_Sensitive','Fat_Suppression'],ascending=False,na_position='last')
                chosen[plane]=str(subset.iloc[0]['SeriesInstanceUID'])
        d[str(uid)]=chosen
    return d
tr_index=index_series(train_series)
te_index=index_series(test_series)
print('Example series selection:', next(iter(tr_index.items())))

## 5. MRI decoding and normalization

Sample three slices per plane, giving nine grayscale channels for each study. Use DICOM slice position when present and apply robust per-slice percentile normalization. Missing or undecodable series become zero channels; decoding errors are counted.

In [ ]:
from PIL import Image
from functools import lru_cache
DECODE_ERRORS={'count':0}

@lru_cache(maxsize=8192)
def files_for_series(split, study_id, series_id):
    folder=ROOT/f'{split}_series'/study_id/series_id
    files=list(folder.glob('*.dcm'))
    if not files: return ()
    # Use header position where available, fall back to InstanceNumber, then name.
    def sort_key(path):
        try:
            ds=pydicom.dcmread(str(path),stop_before_pixels=True,specific_tags=['ImagePositionPatient','InstanceNumber'])
            if 'ImagePositionPatient' in ds: return (0,float(ds.ImagePositionPatient[-1]),str(path))
            if 'InstanceNumber' in ds: return (1,float(ds.InstanceNumber),str(path))
        except Exception: pass
        return (2,0,str(path))
    return tuple(map(str,sorted(files,key=sort_key)))

def load_slice(filename):
    try:
        ds=pydicom.dcmread(filename)
        a=np.asarray(ds.pixel_array,dtype=np.float32)
        if a.ndim==3: a=a[0]  # handles rare multi-frame slices conservatively
        if a.ndim!=2: raise ValueError('Not a 2D grayscale slice')
        a=a*float(getattr(ds,'RescaleSlope',1))+float(getattr(ds,'RescaleIntercept',0))
        a=np.nan_to_num(a,nan=0.0,posinf=0.0,neginf=0.0)
        lo,hi=np.percentile(a,[1,99])
        a=np.clip((a-lo)/max(hi-lo,1e-5),0,1)
        if str(getattr(ds,'PhotometricInterpretation',''))=='MONOCHROME1': a=1-a
        a=np.asarray(Image.fromarray(np.uint8(a*255)).resize((IMAGE_SIZE,IMAGE_SIZE),resample=Image.Resampling.BILINEAR),dtype=np.float32)/255.
        return a
    except Exception:
        DECODE_ERRORS['count']+=1
        return np.zeros((IMAGE_SIZE,IMAGE_SIZE),np.float32)

def load_study(split,uid,index):
    channels=[]
    for plane in ('Sagittal','Coronal','Axial'):
        sid=index.get(uid,{}).get(plane)
        fs=files_for_series(split,uid,sid) if sid else ()
        if not fs: channels.extend([np.zeros((IMAGE_SIZE,IMAGE_SIZE),np.float32) for _ in range(MAX_FILES_PER_SERIES)]);continue
        select=np.linspace(0,len(fs)-1,MAX_FILES_PER_SERIES).astype(int)
        channels.extend([load_slice(fs[i]) for i in select])
    return np.stack(channels).astype(np.float32)

class KneeDataset(Dataset):
    def __init__(self,df,split,index,augmentation=False):
        self.df=df.reset_index(drop=True); self.split=split;self.index=index;self.augmentation=augmentation
    def __len__(self):return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; uid=str(r['StudyInstanceUID'])
        x=load_study(self.split,uid,self.index)
        if self.augmentation and random.random()<0.5: x=x[:,:,::-1].copy()
        if self.augmentation and random.random()<0.5: x=x[:,::-1,:].copy()
        if all(c in self.df.columns for c in LABELS):
            y=r[LABELS].to_numpy(dtype=np.float32)
            mask=np.isfinite(y).astype(np.float32)
            y=np.nan_to_num(y,nan=0.0)
        else:y=np.zeros(12,np.float32);mask=np.zeros(12,np.float32)
        return torch.from_numpy(x.copy()),torch.from_numpy(y),torch.from_numpy(mask)

tr_ds=KneeDataset(tr_df,'train',tr_index,True)
va_ds=KneeDataset(va_df,'train',tr_index)
print('Study tensor shape:',tr_ds[0][0].shape,'| decode failures:',DECODE_ERRORS['count'])

## 6. PyTorch 2.5D CNN and masked BCE loss

This compact CNN trains from scratch, so it works with Internet disabled. Masked binary cross entropy ignores targets without labels.

In [ ]:
class KneeCNN(nn.Module):
    def __init__(self):
        super().__init__()
        layers=[]; prev=9
        for width in [32,64,128,192]:
            layers += [nn.Conv2d(prev,width,3,padding=1,bias=False),nn.BatchNorm2d(width),nn.SiLU(),
                       nn.Conv2d(width,width,3,padding=1,bias=False),nn.BatchNorm2d(width),nn.SiLU(),
                       nn.MaxPool2d(2)]
            prev=width
        self.features=nn.Sequential(*layers,nn.AdaptiveAvgPool2d(1))
        self.head=nn.Sequential(nn.Flatten(),nn.Dropout(.3),nn.Linear(192,12))
    def forward(self,x):return self.head(self.features(x))

def masked_loss(logits,y,mask):
    bce=nn.functional.binary_cross_entropy_with_logits(logits,y,reduction='none')
    return (bce*mask).sum()/mask.sum().clamp(min=1)

def macro_auc(y,p,mask):
    metrics={}
    for j,label in enumerate(LABELS):
        keep=mask[:,j]>0
        metrics[label]=float(roc_auc_score(y[keep,j],p[keep,j])) if keep.sum()>1 and len(np.unique(y[keep,j]))==2 else np.nan
    valid=[v for v in metrics.values() if np.isfinite(v)]
    return float(np.mean(valid)) if valid else np.nan,metrics

train_loader=DataLoader(tr_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=(DEVICE=='cuda'))
val_loader=DataLoader(va_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=(DEVICE=='cuda'))
model=KneeCNN().to(DEVICE)
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
scaler=torch.amp.GradScaler('cuda',enabled=DEVICE=='cuda')
print('Model parameters:',sum(p.numel() for p in model.parameters()))

## 7. Train and validate

Checkpoint is saved in `/kaggle/working`. If validation AUC is not defined, select by validation loss instead. An elapsed-time limit avoids accidentally spending the entire Kaggle submission runtime on training.

In [ ]:
best_score=-1e9
checkpoint=OUT_DIR/'knee_baseline_best.pt'
start=time.monotonic()
history=[]
for epoch in range(EPOCHS):
    if time.monotonic()-start>TRAIN_SECONDS_BUDGET:
        print('Stopping at configured training time budget'); break
    model.train(); train_losses=[]
    for x,y,m in tqdm(train_loader,desc=f'Epoch {epoch+1}/{EPOCHS}',leave=False):
        x,y,m=x.to(DEVICE),y.to(DEVICE),m.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE,enabled=DEVICE=='cuda'):
            pred=model(x);loss=masked_loss(pred,y,m)
        scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()
        train_losses.append(float(loss.item()))
    model.eval(); ps=[];ys=[];ms=[];v_losses=[]
    with torch.inference_mode():
        for x,y,m in val_loader:
            x=x.to(DEVICE);y=y.to(DEVICE);m=m.to(DEVICE)
            logits=model(x);v_losses.append(float(masked_loss(logits,y,m).item()))
            ps.append(torch.sigmoid(logits).cpu().numpy());ys.append(y.cpu().numpy());ms.append(m.cpu().numpy())
    ps=np.concatenate(ps);ys=np.concatenate(ys);ms=np.concatenate(ms)
    auc,per_target=macro_auc(ys,ps,ms)
    vl=float(np.mean(v_losses)); score=auc if np.isfinite(auc) else -vl
    history.append({'epoch':epoch+1,'train_loss':np.mean(train_losses),'val_loss':vl,'macro_auc':auc})
    print(f'Epoch {epoch+1}: train_loss={np.mean(train_losses):.4f} val_loss={vl:.4f} macro_AUC={auc:.4f}')
    if score>best_score:
        best_score=score
        torch.save({'state_dict':model.state_dict(),'labels':LABELS,'size':IMAGE_SIZE},checkpoint)
assert checkpoint.is_file(), 'Checkpoint not saved'
display(pd.DataFrame(history))
print('Saved:',checkpoint,'| DICOM decode failures:',DECODE_ERRORS['count'])

## 8. Predict every test study

No report fields or expert labels enter inference. During Kaggle scoring, the small visible example test set is replaced by the hidden test set. This cell deliberately processes every row.

In [ ]:
saved=torch.load(checkpoint,map_location='cpu',weights_only=False)
model.load_state_dict(saved['state_dict']);model.to(DEVICE);model.eval()
test_ds=KneeDataset(test[['StudyInstanceUID']],'test',te_index,False)
test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=DEVICE=='cuda')
probs=[]
with torch.inference_mode():
    for x,_,_ in tqdm(test_loader,desc='Test inference'):
        x=x.to(DEVICE)
        probs.append(torch.sigmoid(model(x)).cpu().numpy())
probs=np.concatenate(probs,axis=0)
assert probs.shape==(len(test),12)
print('Predictions:',probs.shape,'| decode failures:',DECODE_ERRORS['count'])

## 9. Write and verify submission.csv

In [ ]:
prediction_df=pd.DataFrame(probs,columns=LABELS)
prediction_df.insert(0,'StudyInstanceUID',test['StudyInstanceUID'].to_numpy())
submission=sample[['StudyInstanceUID']].merge(prediction_df,on='StudyInstanceUID',how='left',validate='one_to_one',sort=False)
assert len(submission)==len(sample),'Submission row count mismatch'
assert submission['StudyInstanceUID'].tolist()==sample['StudyInstanceUID'].tolist(), 'Study ID order mismatch'
assert submission[LABELS].notna().all().all(), 'Missing prediction(s)'
assert np.isfinite(submission[LABELS].to_numpy()).all(), 'Non-finite prediction(s)'
assert ((submission[LABELS]>=0)&(submission[LABELS]<=1)).all().all(), 'Predictions outside [0,1]'
assert submission.columns.tolist()==sample.columns.tolist(), 'Wrong submission columns'
submission_path=OUT_DIR/'submission.csv'
submission.to_csv(submission_path,index=False)
print('Submission ready:',submission_path,'| rows:',len(submission),'| bytes:',submission_path.stat().st_size)
display(submission.head())

## Next improvements

1. Validate the weak-label rules manually against several hundred reports, including non-English languages and negation.
2. Train using more studies, stronger pretrained backbones added as offline Kaggle datasets, and per-series features with attention pooling.
3. Introduce scanner/site-aware validation to estimate domain generalization.
4. Consider image-to-text report supervision or teacher-student pseudo-labeling, keeping **reports out of test-time inputs**.
5. Cache normalized DICOM slices between runs to accelerate experiments.

**Submission:** In Kaggle, attach the competition data, disable Internet, select GPU, run all cells, then select **Save Version → Save & Run All** and submit the resulting `submission.csv`.